# 01 — Chunking Strategies

**Repo:** ai-learning-notes | **Folder:** rag-and-retrieval  
**Built alongside:** local-rag-llm (github.com/RT91-data/local-rag-llm)

---

## What this notebook covers

- Why chunking is the most impactful decision in a RAG system
- How RecursiveCharacterTextSplitter actually works internally
- Character count vs token count — why they differ and why it matters
- Overlap mechanics — the gain and the cost
- Real failure: how chunk_size=1000 broke Q1 in the RAGAS eval
- Semantic chunking — splitting on meaning, not character count
- Parent-document retrieval pattern
- Decision guide: which strategy for which use case

**This is an explanatory notebook.** Pure Python only — no external dependencies.

---
## 1. Why Chunking Is the Highest-Leverage Decision

Most RAG tutorials treat chunking as a footnote — pick chunk_size=1000 and move on.  
In practice, chunking decisions determine whether your RAG answers correctly more than any other single factor.

**Here's why:**

The retriever can only return what's in a chunk.  
If the answer is split across two chunks, and only one chunk is retrieved, the answer is incomplete.  
No amount of model quality, prompt engineering, or reranking fixes a missing chunk.

**Real evidence from local-rag-llm RAGAS evaluation:**

| chunk_size | Question | Context Recall | Root Cause |
|---|---|---|---|
| 1000 | "What are the 7 pillars?" | 0.286 | Pillars 3–6 split across chunk boundary |
| 2000 | "What are the 7 pillars?" | 0.857 | Most pillars in one chunk, Pillar 6 still truncated |
| 2500 (projected) | "What are the 7 pillars?" | ~1.0 | All pillars fit in one chunk |

The embedding model, the LLM, and the reranker were identical across these runs.  
Only chunk_size changed. Context recall tripled.

---
## 2. Fixed-Size Chunking — The Naive Approach

The simplest chunking strategy: split every N characters, step forward by N-overlap.

**Problems:**
- Cuts mid-sentence, mid-word, mid-table
- No awareness of document structure
- Overlap helps but doesn't fully compensate

Fixed-size chunking is almost never the right choice for production RAG.

In [ ]:
def fixed_size_chunk(text, chunk_size=100, overlap=20):
    """Naive fixed-size chunker. Cuts anywhere."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start += chunk_size - overlap
    return chunks


text = (
    "Pillar 5 - Identity and Access Management (IAM): Identity Administrators are tasked "
    "with cryptographically verifying exactly who is interacting with the system. The major "
    "risk is the Confused Deputy problem, where an over-privileged agent is tricked into "
    "executing unauthorised commands."
)

chunks = fixed_size_chunk(text, chunk_size=120, overlap=0)

print("FIXED-SIZE CHUNKING (no overlap, size=120)")
print("=" * 60)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: '{chunk}'")
    print()

print("Problem: 'Confused Deputy problem' is split across chunk 1 and chunk 2.")
print("If only chunk 1 is retrieved, the answer is incomplete.")

---
## 3. RecursiveCharacterTextSplitter — How It Actually Works

LangChain's `RecursiveCharacterTextSplitter` is what `local-rag-llm` uses.  
It is **not** a fixed-size splitter. It tries a hierarchy of separators, from coarsest to finest:

```
Default separator hierarchy:
  1. "\n\n"   (double newline — paragraph boundary)
  2. "\n"     (single newline — line boundary)
  3. " "      (space — word boundary)
  4. ""       (character — last resort)
```

**The algorithm:**
1. Try to split on `\n\n` first
2. If any resulting piece is still larger than `chunk_size`, recursively split that piece using `\n`
3. If still too large, split on spaces
4. If still too large, split on characters
5. After splitting, merge small adjacent pieces back together until they're close to `chunk_size`

The result: chunks that respect paragraph and sentence boundaries whenever possible,  
only falling back to mid-sentence cuts when absolutely necessary.

In [ ]:
def recursive_chunk(text, chunk_size=200, overlap=40, separators=None):
    """
    Simplified RecursiveCharacterTextSplitter.
    Illustrates the recursive separator logic — not production grade.
    """
    if separators is None:
        separators = ["\n\n", "\n", " ", ""]

    def split_text(text, separators):
        if not separators:
            return [text]

        separator = separators[0]
        remaining = separators[1:]

        if separator == "":
            splits = list(text)
        else:
            splits = text.split(separator)

        chunks = []
        current = ""

        for split in splits:
            if not split.strip():
                continue
            candidate = (current + separator + split).strip() if current else split

            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    chunks.append(current)
                if len(split) > chunk_size:
                    # Recursively split this piece with the next separator
                    sub_chunks = split_text(split, remaining)
                    chunks.extend(sub_chunks[:-1])
                    current = sub_chunks[-1] if sub_chunks else ""
                else:
                    current = split

        if current:
            chunks.append(current)

        return chunks

    return split_text(text, separators)


document = """Pillar 1 - Infrastructure & Networking
Cloud Infrastructure Engineers must secure the foundational environment against upstream poisoning and container escapes.

Pillar 2 - Data
Data Architects face the threat of agents leaking sensitive information from their context windows or ingesting poisoned RAG data.

Pillar 3 - Model
AI Engineers must defend the core application logic against semantic attacks that subvert the model's instructions."""

chunks = recursive_chunk(document, chunk_size=180, overlap=0)

print("RECURSIVE CHUNKING — respects paragraph boundaries")
print("=" * 60)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i} ({len(chunk)} chars):")
    print(f"  '{chunk[:100]}{'...' if len(chunk) > 100 else ''}'")
    print()

print("Each pillar stays intact — no mid-sentence cuts.")
print("Compare to fixed-size which would cut wherever the counter hits 180.")

---
## 4. Character Count vs Token Count

`chunk_size=1000` means 1000 **characters**, not 1000 **tokens**.  
LLMs measure context in tokens, not characters.

**The gap:**
- Simple English: ~1 token per 4 characters → 1000 chars ≈ 250 tokens
- Technical text with long words: ~1 token per 3 characters → 1000 chars ≈ 333 tokens  
- Code: ~1 token per 2-3 characters → 1000 chars ≈ 333-500 tokens

**Why this matters:**
- Your chunk_size is a character budget, but the LLM's context window is a token budget
- Sending 5 chunks × 1000 chars = ~5000 chars ≈ 1250 tokens — well within limits
- But if your document is dense technical content, those same 5 chunks might be 2500 tokens
- Not a crisis at typical RAG scale, but important when designing for very large context payloads

In [ ]:
# Illustrating character vs token count differences
# Real tokenisation requires tiktoken or the model's tokeniser
# This approximates using known ratios

samples = [
    (
        "simple english",
        "The cat sat on the mat and looked at the dog that ran past the old red barn.",
        4.0  # avg chars per token for simple English
    ),
    (
        "technical security text",
        "Zero Ambient Authority enforces Just-In-Time cryptographic downscoping via SPIFFE IDs and ABAC permissions matrix.",
        3.2  # technical terms have more chars per token
    ),
    (
        "python code",
        "def retrieve_with_threshold(vectorstore, query: str, k: int = 8, threshold: float = 0.3) -> list:",
        2.8  # code has many short tokens
    ),
    (
        "d365 erp data",
        "VendTable.AccountNum: CONTOSO-001 | PaymTermId: NET30 | CreditMax: 50000.00 | CurrencyCode: SGD",
        2.5  # IDs, codes, numbers are each their own token
    ),
]

print(f"{'Text type':<25} {'Chars':>7} {'Est tokens':>12} {'Chars/token':>12}")
print("-" * 60)
for text_type, text, chars_per_token in samples:
    chars = len(text)
    tokens = int(chars / chars_per_token)
    print(f"{text_type:<25} {chars:>7} {tokens:>12} {chars_per_token:>12.1f}")

print()
print("Implication for local-rag-llm (chunk_size=2000):")
print(f"  Simple English:  2000 chars ≈ {2000//4:,} tokens per chunk")
print(f"  Technical text:  2000 chars ≈ {2000//3:,} tokens per chunk")
print(f"  5 chunks × 667 tokens = {5*667:,} tokens sent to Claude per query")
print(f"  Claude Sonnet limit: 200,000 tokens — well within budget")

---
## 5. Overlap — What You Gain and What You Pay

Overlap means the end of chunk N is repeated at the start of chunk N+1.

**What you gain:**
- Answers that span a chunk boundary are partially present in both adjacent chunks
- Either chunk can serve as a complete-enough answer even if the other isn't retrieved
- Continuity: the embedding of chunk N+1 carries context from the end of chunk N

**What you pay:**
- Storage: more chunks means larger FAISS index
- Retrieval noise: the same sentence can appear in two chunks, both retrieved, wasting context slots
- Deduplication complexity: your merge step needs to catch duplicate content

**Rule of thumb:** overlap = 20% of chunk_size.  
At chunk_size=2000, overlap=400 is correct.  
Going above 50% overlap is almost never justified.

In [ ]:
def chunker_with_overlap(text, chunk_size=150, overlap=50):
    """Character-based chunker with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append({
            "text": text[start:end],
            "start": start,
            "end": end
        })
        if end == len(text):
            break
        start += chunk_size - overlap
    return chunks


text = (
    "The Green Team executes a Stateful Quarantine via SOAR playbooks. "
    "This gracefully revokes the agent's specific tool access, freezing its ability to act. "
    "It preserves short-term memory intact for forensic analysis."
)

print("WITH OVERLAP (chunk_size=120, overlap=40)")
print("=" * 60)
chunks = chunker_with_overlap(text, chunk_size=120, overlap=40)
for i, c in enumerate(chunks):
    print(f"Chunk {i} [chars {c['start']}-{c['end']}]:")
    print(f"  '{c['text']}'")
    print()

# Show the overlapping region
if len(chunks) >= 2:
    overlap_text = chunks[1]['text'][:40]
    print(f"Overlap region (start of chunk 1): '{overlap_text}'")
    print(f"This text also appears at end of chunk 0 — that's the overlap.")

print()
print("WITHOUT OVERLAP (same text)")
print("=" * 60)
chunks_no_overlap = chunker_with_overlap(text, chunk_size=120, overlap=0)
for i, c in enumerate(chunks_no_overlap):
    print(f"Chunk {i}: '{c['text']}'")

In [ ]:
# Storage cost of overlap

def estimate_chunk_count(doc_chars, chunk_size, overlap):
    step = chunk_size - overlap
    return max(1, (doc_chars - overlap) // step)

doc_chars = 100_000  # typical 50-page PDF

configs = [
    (1000, 0,   "No overlap"),
    (1000, 200, "20% overlap"),
    (1000, 500, "50% overlap"),
    (2000, 400, "local-rag-llm current"),
    (2000, 0,   "2000 no overlap"),
]

print(f"{'Config':<30} {'chunk_size':>10} {'overlap':>8} {'# chunks':>10} {'storage x':>10}")
print("-" * 75)
baseline = None
for label, size, overlap, name in [(c[2], c[0], c[1], c[2]) for c in configs]:
    n = estimate_chunk_count(doc_chars, size, overlap)
    if baseline is None:
        baseline = n
    print(f"{name:<30} {size:>10} {overlap:>8} {n:>10} {n/baseline:>10.2f}x")

print()
print("50% overlap nearly doubles your index size.")
print("local-rag-llm uses 20% overlap — good balance of boundary coverage vs storage.")

---
## 6. The Real Failure: Multi-Point Answers Spanning Chunk Boundaries

The most common chunking failure in practice is **enumerated content that spans pages**.

Examples:
- "List the 7 pillars of agent security" — answer is 3 pages long
- "What are the evaluation dimensions?" — 7 items across 2 pages  
- "What does the security recap recommend?" — 5 bullet points across a page break

**What happens with chunk_size=1000:**
```
Chunk A (page 10):  Pillar 1 + Pillar 2
Chunk B (page 11):  Pillar 3 + Pillar 4 + Pillar 5 + Pillar 6 (truncated)
Chunk C (page 12):  Pillar 7 + closing paragraph
```
FAISS retrieves chunk A and chunk C (both match "7 pillars").  
Chunk B is ranked lower (it doesn't contain "7 pillars" header) and gets dropped after reranking.  
The answer is missing Pillars 3–6.

**Three fixes, in order of complexity:**
1. Increase chunk_size until the content fits (simple, works often)
2. Add a junk chunk filter so TOC/header chunks don't crowd out content chunks
3. Use parent-document retrieval (most robust, covered below)

In [ ]:
# Simulating the 7-pillars retrieval failure

# Fake similarity scores (what FAISS would return)
candidate_chunks = [
    {
        "id": "A",
        "content": "The Foundation: The 7-Pillar Agent Security Architecture. Pillar 1 - Infrastructure. Pillar 2 - Data.",
        "faiss_score": 0.91,  # high: contains '7-Pillar' header
        "page": 10
    },
    {
        "id": "B",
        "content": "Pillar 3 - Model. Pillar 4 - Application & Runtime. Pillar 5 - IAM. Pillar 6 - Observability.",
        "faiss_score": 0.62,  # lower: no header, just content
        "page": 11
    },
    {
        "id": "C",
        "content": "Pillar 7 - Governance. This 7-pillar architecture provides the universal baseline.",
        "faiss_score": 0.84,  # high: contains '7-pillar' text
        "page": 12
    },
    {
        "id": "TOC",
        "content": "Table of contents. The 7-Pillar Agent Security Architecture...9. Sandboxes...13.",
        "faiss_score": 0.88,  # high: contains exact header text
        "page": 3
    },
]

# Sort by FAISS score, take top 3 (simulating top_k=3)
ranked = sorted(candidate_chunks, key=lambda x: x['faiss_score'], reverse=True)

print("FAISS RETRIEVAL — top 3 chunks")
print("=" * 60)
print(f"{'Rank':<6} {'ID':<6} {'Page':<6} {'Score':<8} Content (truncated)")
print("-" * 75)
for i, chunk in enumerate(ranked[:3], 1):
    print(f"{i:<6} {chunk['id']:<6} {chunk['page']:<6} {chunk['faiss_score']:<8.2f} {chunk['content'][:50]}...")

print()
print(f"Chunk B (page 11, Pillars 3-6) ranked #{[c['id'] for c in ranked].index('B')+1} — NOT in top 3.")
print("TOC chunk took its slot despite containing no actual pillar descriptions.")
print()
print("Result: Answer contains Pillars 1, 2, 7 but MISSING Pillars 3, 4, 5, 6.")
print("RAGAS context_recall = 0.286 — matches the actual eval result.")

---
## 7. Semantic Chunking

Instead of splitting on character count or separators, semantic chunking splits on **meaning shifts**.

**How it works:**
1. Split the document into sentences
2. Embed each sentence
3. Compute cosine similarity between adjacent sentences
4. When similarity drops significantly, insert a chunk boundary
5. The boundary marks a topic transition

**Pros:**
- Chunks naturally contain one coherent topic
- No arbitrary cuts mid-thought
- Retrieval precision improves — each chunk is about one thing

**Cons:**
- Requires an embedding call per sentence — expensive at index time
- Chunk sizes vary widely — some topics need 1 sentence, others need 20
- Threshold tuning: what counts as a "significant" similarity drop?
- Not yet in `local-rag-llm` — on the roadmap

**When to use:**  
Documents with clearly distinct sections (research papers, legal documents, D365 module documentation).  
Less useful for narrative documents where topics blend gradually.

In [ ]:
import math

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x**2 for x in a))
    norm_b = math.sqrt(sum(x**2 for x in b))
    if norm_a == 0 or norm_b == 0:
        return 0
    return dot / (norm_a * norm_b)


# Simulated sentence-level embeddings (2D for illustration)
# Real embeddings would be 768D from nomic-embed-text
sentences = [
    ("Pillar 1 covers infrastructure and networking security.",           [0.9, 0.1]),
    ("Ephemeral sandboxes isolate code execution from the host.",         [0.85, 0.15]),
    ("Network egress governance prevents data exfiltration.",             [0.8, 0.2]),
    # Topic shift to IAM
    ("Pillar 5 covers identity and access management.",                   [0.1, 0.9]),
    ("SPIFFE IDs provide cryptographic identity for each agent.",         [0.15, 0.85]),
    ("JIT downscoping ensures credentials expire after each task.",       [0.2, 0.8]),
]

SPLIT_THRESHOLD = 0.5  # split when similarity drops below this

print("SEMANTIC CHUNKING — similarity between adjacent sentences")
print("=" * 65)
print(f"{'#':<4} {'Sentence (truncated)':<48} {'Sim to prev':>12}")
print("-" * 65)

chunks = []
current_chunk = [sentences[0][0]]

for i, (sentence, vec) in enumerate(sentences):
    if i == 0:
        print(f"{i:<4} {sentence[:47]:<48} {'(first)':>12}")
        continue

    prev_vec = sentences[i-1][1]
    sim = cosine_similarity(vec, prev_vec)
    split_marker = " ← SPLIT" if sim < SPLIT_THRESHOLD else ""

    print(f"{i:<4} {sentence[:47]:<48} {sim:>12.3f}{split_marker}")

    if sim < SPLIT_THRESHOLD:
        chunks.append(current_chunk)
        current_chunk = [sentence]
    else:
        current_chunk.append(sentence)

chunks.append(current_chunk)

print()
print(f"Result: {len(chunks)} semantic chunks")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i}: {len(chunk)} sentences — topic: {'Infrastructure' if i==0 else 'IAM'}")

---
## 8. Parent-Document Retrieval Pattern

The best solution for the multi-point answer problem is the **parent-document retrieval pattern**.

**The insight:**  
- Small chunks → better retrieval precision (specific, focused embedding)
- Large chunks → better answer completeness (more context for the LLM)
- You want both — retrieve with small, answer with large

**How it works:**
1. Store large parent chunks (e.g., full pages) in a document store
2. Create small child chunks from each parent (e.g., paragraph-sized)
3. Embed and index only the child chunks
4. At query time: retrieve relevant child chunks by embedding similarity
5. For each retrieved child, fetch its full parent chunk
6. Send the parent chunks to the LLM — not the child chunks

**Result:**  
Retrieval precision of a small chunk.  
Answer completeness of a large chunk.

For the "7 pillars" question:  
- Child chunk "Pillar 3 - Model..." gets retrieved (specific, focused)
- Its parent (the full page 11) is returned — containing Pillars 3, 4, 5, and 6
- Context recall → 1.0

In [ ]:
# Illustrating the parent-document pattern

# Step 1: Large parent chunks (one per page or section)
parent_chunks = {
    "page_10": (
        "[Page 10] The Foundation: The 7-Pillar Agent Security Architecture. "
        "Pillar 1 - Infrastructure & Networking: Cloud Infrastructure Engineers must secure "
        "the foundational environment. "
        "Pillar 2 - Data: Data Architects face the threat of agents leaking sensitive information."
    ),
    "page_11": (
        "[Page 11] "
        "Pillar 3 - Model: AI Engineers must defend against semantic attacks. "
        "Pillar 4 - Application & Runtime: Deploy LLM firewalls and Agent Gateways. "
        "Pillar 5 - IAM: Assign SPIFFE IDs and use JIT downscoping. "
        "Pillar 6 - Observability & Security Ops: Combat invisible failures."
    ),
    "page_12": (
        "[Page 12] "
        "Pillar 7 - Governance: Ensure regulatory compliance including EU AI Act. "
        "This 7-pillar architecture provides the universal baseline."
    ),
}

# Step 2: Small child chunks (what gets embedded and indexed)
child_chunks = [
    {"id": "c1", "parent_id": "page_10", "text": "Pillar 1 - Infrastructure & Networking: isolate runtime in ephemeral kernel-level sandboxes."},
    {"id": "c2", "parent_id": "page_10", "text": "Pillar 2 - Data: protect context with CMEK and mTLS. Enforce tenant partitioning."},
    {"id": "c3", "parent_id": "page_11", "text": "Pillar 3 - Model: treat system instructions as cryptographically attested artifacts."},
    {"id": "c4", "parent_id": "page_11", "text": "Pillar 4 - Application & Runtime: LLM firewalls and Centralised Agent Gateways."},
    {"id": "c5", "parent_id": "page_11", "text": "Pillar 5 - IAM: SPIFFE IDs, ABAC, JIT downscoping. Intent × User × Time matrix."},
    {"id": "c6", "parent_id": "page_11", "text": "Pillar 6 - Observability: Red, Blue, Green SecOps triad via OpenTelemetry."},
    {"id": "c7", "parent_id": "page_12", "text": "Pillar 7 - Governance: EU AI Act, Logic Reviews, Risk-Stratified Attestation."},
]

# Step 3: Query retrieves child chunks (simulated)
def fake_retrieve_children(query, child_chunks, top_k=3):
    """Simulate retrieval — in reality this is FAISS similarity search."""
    # All pillars are relevant to '7 pillars' query
    return child_chunks[:top_k]

query = "What are the 7 pillars of the agent security architecture?"
retrieved_children = fake_retrieve_children(query, child_chunks, top_k=3)

# Step 4: Fetch parents of retrieved children
retrieved_parent_ids = list(dict.fromkeys(c["parent_id"] for c in retrieved_children))
retrieved_parents = {pid: parent_chunks[pid] for pid in retrieved_parent_ids}

print("PARENT-DOCUMENT RETRIEVAL PATTERN")
print("=" * 60)
print(f"Query: '{query}'")
print()
print("Step 1 — Child chunks retrieved (small, precise):")
for child in retrieved_children:
    print(f"  [{child['id']} → parent:{child['parent_id']}] {child['text'][:60]}...")

print()
print("Step 2 — Parent chunks fetched (large, complete):")
for pid, ptext in retrieved_parents.items():
    print(f"  [{pid}] {len(ptext)} chars — {ptext[:80]}...")

print()
print("What gets sent to the LLM: the PARENT chunks (complete page content)")
print("What was used for retrieval: the CHILD chunks (focused paragraph)")
print("Result: retrieval precision of a small chunk, answer quality of a large chunk.")

---
## 9. Chunking Strategy Decision Guide

| Situation | Recommended Strategy | Why |
|---|---|---|
| General document Q&A | RecursiveCharacterTextSplitter, chunk_size=1500-2500 | Good default, respects boundaries |
| Answers span multiple pages | Increase chunk_size OR parent-document pattern | Ensures full answer is in one chunk |
| Dense technical docs (D365 help) | Semantic chunking | Topics are clearly bounded |
| Structured records (ERP transactions) | No chunking — one record = one document | Records are already the right granularity |
| Mixed content (text + tables) | pdfplumber + RecursiveCharacterTextSplitter | Tables need special extraction |
| High retrieval precision needed | Parent-document pattern | Small child = precise embedding |
| Fast prototyping | chunk_size=1000, overlap=200 | Standard default, tune later |

**The sequence for tuning chunk_size:**
1. Start with chunk_size=1000, overlap=200
2. Run RAGAS evaluation — check context_recall per question
3. For questions with recall < 0.8, check which chunks were retrieved
4. If the answer was split across chunks → increase chunk_size
5. If wrong chunks were retrieved → consider semantic chunking or parent-document
6. Re-run RAGAS — compare recall
7. Stop when recall is acceptable or you hit the token budget ceiling

In [ ]:
# Summary: chunk_size impact on local-rag-llm RAGAS results

results = [
    {
        "config": "chunk_size=1000, overlap=200",
        "q1_7pillars_recall": 0.286,
        "avg_recall_20q": 0.912,
        "note": "Pillars 3-6 split across boundary"
    },
    {
        "config": "chunk_size=2000, overlap=400",
        "q1_7pillars_recall": 0.857,
        "avg_recall_20q": 0.939,
        "note": "Pillar 6 still truncated at end of chunk"
    },
    {
        "config": "chunk_size=2500, overlap=500 (projected)",
        "q1_7pillars_recall": 1.000,
        "avg_recall_20q": 0.960,
        "note": "All pillars fit — full Pillar 6 included"
    },
    {
        "config": "parent-document pattern (projected)",
        "q1_7pillars_recall": 1.000,
        "avg_recall_20q": 0.975,
        "note": "Best precision + completeness"
    },
]

print(f"{'Config':<45} {'Q1 recall':>10} {'Avg recall':>12} Note")
print("-" * 100)
for r in results:
    flag = " ← current" if "2000" in r["config"] else ""
    print(
        f"{r['config']:<45} "
        f"{r['q1_7pillars_recall']:>10.3f} "
        f"{r['avg_recall_20q']:>12.3f} "
        f"{r['note']}{flag}"
    )

print()
print("Key takeaway: going from 1000 to 2000 chars nearly tripled Q1 recall.")
print("A 2500 char chunk would solve it completely.")
print("Parent-document pattern gives the most robust solution.")

---
## 10. Summary

| Concept | Key point |
|---|---|
| **Fixed-size chunking** | Simple but cuts anywhere — never use in production |
| **RecursiveCharacterTextSplitter** | Respects paragraph → sentence → word → character hierarchy |
| **chunk_size** | Character budget — tune based on RAGAS context_recall |
| **overlap** | 20% of chunk_size is the standard — reduces boundary split misses |
| **chars vs tokens** | ~3-4 chars per token for English text — budget accordingly |
| **Semantic chunking** | Splits on meaning shifts — better precision, expensive at index time |
| **Parent-document pattern** | Retrieve with small chunks, answer with large parents — best of both |
| **When to change chunk_size** | When RAGAS context_recall is low for multi-point answers |

**The most important rule:**  
Never set chunk_size by intuition. Set it by RAGAS context_recall.  
A question with recall < 0.8 is telling you the answer was split across chunks.

---
## Next Notebooks

- **02** — Embeddings and vector stores (FAISS internals, distance metrics, index types)
- **03** — Sparse retrieval and BM25 (how keyword search works, when it beats dense)
- **04** — Hybrid retrieval (EnsembleRetriever, weighting, Reciprocal Rank Fusion)